# Storage Diagnostics Admin Tool

Ferramenta administrativa para investigação de consumo excessivo de storage no Databricks/Unity Catalog.

## Objetivo
Identificar as principais causas de consumo de storage, incluindo:
- **Tabelas grandes sem política de retenção** — dados históricos acumulados sem limpeza
- **Bloat de versões Delta** — muitas versões antigas retidas (falta de VACUUM)
- **Arquivos temporários e intermediários** — tabelas de staging, temp, ou ETL que permanecem após o processamento
- **Logs, checkpoints e metadados** — artefatos de Structured Streaming e Delta que crescem sem controle
- **Estruturas duplicadas** — tabelas repetidas entre camadas (bronze/silver/gold) ou ambientes (dev/staging/prod)
- **Objetos órfãos** — tabelas ou volumes sem acesso recente que ainda consomem storage
- **Volumes com arquivos pesados** — uploads ou exports esquecidos

## Como usar
1. Configure os parâmetros na célula seguinte (catálogo, schema, thresholds)
2. Execute as células sequencialmente
3. Analise os resultados consolidados na seção final

> ⚠️ **Permissões necessárias**: Este notebook requer acesso de leitura ao `system.information_schema` e ao `system.access` (audit logs). Solicite as permissões ao administrador do workspace se necessário.

In [0]:
# =============================================================================
# LEITURA DOS PARÂMETROS (Widgets do Notebook)
# Os valores são configurados nos widgets acima — sem necessidade de editar código.
# =============================================================================

# Escopo da análise (vazio = todos)
_cat = dbutils.widgets.get("catalog_filter").strip()
_sch = dbutils.widgets.get("schema_filter").strip()
CATALOG_FILTER = _cat if _cat else None
SCHEMA_FILTER = _sch if _sch else None

# Thresholds
TABLE_SIZE_THRESHOLD_GB = int(dbutils.widgets.get("table_size_threshold_gb"))
STALE_DAYS_THRESHOLD = int(dbutils.widgets.get("stale_days_threshold"))
VERSION_COUNT_THRESHOLD = int(dbutils.widgets.get("version_count_threshold"))

# Quantidade de resultados
TOP_N = int(dbutils.widgets.get("top_n"))

# Padrões que indicam objetos temporários/intermediários
TEMP_PATTERNS = [p.strip() for p in dbutils.widgets.get("temp_patterns").split(",") if p.strip()]

print(f"""
╔═══════════════════════════════════════════════════╗
║  STORAGE DIAGNOSTICS - Configuração Ativa         ║
╠═══════════════════════════════════════════════════╣
║  Catálogo: {CATALOG_FILTER or 'TODOS':<39}
║  Schema:   {SCHEMA_FILTER or 'TODOS':<39}
║  Threshold tamanho: {TABLE_SIZE_THRESHOLD_GB} GB{' ':<25}
║  Threshold inatividade: {STALE_DAYS_THRESHOLD} dias{' ':<29}
║  Threshold versões: {VERSION_COUNT_THRESHOLD}{' ':<29}
║  Top N resultados: {TOP_N}{' ':<29}
╚═══════════════════════════════════════════════════╝
""")

## 1. Análise de Tamanho de Tabelas
Identifica as maiores tabelas no ambiente, incluindo tamanho total, número de arquivos e partições.

In [0]:
%sql
-- Inventário completo de tabelas no escopo configurado
-- Esta célula lista TODAS as tabelas elegíveis; o ranking por tamanho vem na célula seguinte (DESCRIBE DETAIL)

SELECT
  table_catalog,
  table_schema,
  table_name,
  table_type,
  data_source_format,
  table_owner,
  created,
  last_altered,
  DATEDIFF(CURRENT_DATE(), last_altered) AS days_since_last_change,
  comment
FROM system.information_schema.tables
WHERE table_type IN ('MANAGED', 'EXTERNAL')
  AND (:catalog_filter = '' OR table_catalog = :catalog_filter)
  AND (:schema_filter = '' OR table_schema = :schema_filter)
ORDER BY last_altered DESC

In [0]:
# Abordagem alternativa usando DESCRIBE DETAIL para obter tamanhos precisos
# Itera sobre as tabelas encontradas no information_schema

from pyspark.sql.functions import col, lit, sum as spark_sum, round as spark_round
import concurrent.futures

# Buscar lista de tabelas
catalog_clause = f"AND table_catalog = '{CATALOG_FILTER}'" if CATALOG_FILTER else ""
schema_clause = f"AND table_schema = '{SCHEMA_FILTER}'" if SCHEMA_FILTER else ""

tables_df = spark.sql(f"""
  SELECT table_catalog, table_schema, table_name
  FROM system.information_schema.tables
  WHERE table_type IN ('MANAGED', 'EXTERNAL')
    AND data_source_format = 'DELTA'
    {catalog_clause}
    {schema_clause}
""")

table_list = tables_df.collect()
print(f"Total de tabelas Delta encontradas: {len(table_list)}")

# Coletar detalhes de cada tabela via DESCRIBE DETAIL
results = []
errors = []

for row in table_list:
    fqn = f"`{row.table_catalog}`.`{row.table_schema}`.`{row.table_name}`"
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {fqn}").collect()[0]
        results.append({
            "catalog": row.table_catalog,
            "schema": row.table_schema,
            "table": row.table_name,
            "full_name": fqn,
            "size_bytes": detail.sizeInBytes if detail.sizeInBytes else 0,
            "num_files": detail.numFiles if detail.numFiles else 0,
            "partitions": str(detail.partitionColumns) if detail.partitionColumns else "[]",
            "created_at": str(detail.createdAt) if detail.createdAt else None,
            "last_modified": str(detail.lastModified) if detail.lastModified else None,
        })
    except Exception as e:
        errors.append({"table": fqn, "error": str(e)[:200]})

print(f"Tabelas processadas com sucesso: {len(results)}")
print(f"Tabelas com erro de acesso: {len(errors)}")

# Criar DataFrame com resultados
if results:
    df_sizes = spark.createDataFrame(results)
    df_sizes = df_sizes.withColumn("size_gb", spark_round(col("size_bytes") / (1024**3), 2)) \
                       .orderBy(col("size_bytes").desc()) \
                       .limit(TOP_N)
    df_sizes.createOrReplaceTempView("table_sizes_detail")
    display(df_sizes.select("catalog", "schema", "table", "size_gb", "num_files", "partitions", "last_modified"))
else:
    print("⚠️ Nenhuma tabela encontrada com os filtros configurados.")

## 2. Bloat de Versões Delta (VACUUM pendente)
Identifica tabelas com muitas versões históricas retidas, indicando que `VACUUM` não está sendo executado adequadamente. Cada versão antiga mantém arquivos Parquet no storage que poderiam ser removidos.

In [0]:
# Analisa o histórico de versões das tabelas Delta
# Tabelas com muitas versões indicam falta de VACUUM

from pyspark.sql.functions import col, count, min as spark_min, max as spark_max, datediff, current_timestamp, lit
from datetime import datetime

version_results = []

for row in table_list:
    fqn = f"`{row.table_catalog}`.`{row.table_schema}`.`{row.table_name}`"
    try:
        history = spark.sql(f"DESCRIBE HISTORY {fqn} LIMIT 1000")
        hist_stats = history.agg(
            count("*").alias("total_versions"),
            spark_min("timestamp").alias("oldest_version"),
            spark_max("timestamp").alias("newest_version")
        ).collect()[0]
        
        # Verificar se há operações VACUUM no histórico
        vacuum_ops = history.filter(col("operation") == "VACUUM END").count()
        
        version_results.append({
            "catalog": row.table_catalog,
            "schema": row.table_schema,
            "table": row.table_name,
            "total_versions": hist_stats.total_versions,
            "oldest_version_date": str(hist_stats.oldest_version)[:19] if hist_stats.oldest_version else None,
            "newest_version_date": str(hist_stats.newest_version)[:19] if hist_stats.newest_version else None,
            "vacuum_operations": vacuum_ops,
            "days_of_history": (hist_stats.newest_version - hist_stats.oldest_version).days if hist_stats.oldest_version and hist_stats.newest_version else 0
        })
    except Exception as e:
        pass  # Tabelas sem permissão ou não-Delta são ignoradas

if version_results:
    df_versions = spark.createDataFrame(version_results)
    
    # Filtrar tabelas com versões acima do threshold
    df_version_bloat = df_versions.filter(col("total_versions") >= VERSION_COUNT_THRESHOLD) \
        .orderBy(col("total_versions").desc()) \
        .limit(TOP_N)
    
    df_version_bloat.createOrReplaceTempView("version_bloat")
    
    total_problematic = df_version_bloat.count()
    print(f"⚠️ {total_problematic} tabelas com mais de {VERSION_COUNT_THRESHOLD} versões retidas:")
    print(f"   Essas tabelas provavelmente precisam de VACUUM para liberar espaço.")
    print(f"   Tabelas SEM nenhum VACUUM no histórico são as mais críticas.\n")
    display(df_version_bloat)
else:
    print("✅ Nenhuma tabela com bloat de versões significativo encontrada.")

## 3. Análise de Volumes
Identifica volumes do Unity Catalog com maior consumo de storage. Volumes podem acumular arquivos de upload, exports, datasets temporários e artefatos esquecidos.

In [0]:
# Análise de volumes - identifica volumes com maior consumo

from pyspark.sql.functions import col, round as spark_round

# --- Funções auxiliares ---

def _get_volume_size_recursive(path):
    """Calcula tamanho total de um volume recursivamente (inclui subdiretórios)."""
    total_size = 0
    file_count = 0
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return -1, -1
    for item in items:
        if item.path.endswith("/"):
            sub_size, sub_count = _get_volume_size_recursive(item.path)
            if sub_size >= 0:
                total_size += sub_size
                file_count += sub_count
        else:
            total_size += item.size
            file_count += 1
    return total_size, file_count


# --- Query de metadados de volumes ---

_vol_cat_filter = f"AND volume_catalog = '{CATALOG_FILTER}'" if CATALOG_FILTER else ""
_vol_sch_filter = f"AND volume_schema = '{SCHEMA_FILTER}'" if SCHEMA_FILTER else ""

volume_list = spark.sql(f"""
    SELECT
        volume_catalog,
        volume_schema,
        volume_name,
        volume_type,
        storage_location,
        created AS created_at,
        last_altered AS last_altered_at
    FROM system.information_schema.volumes
    WHERE 1=1
      {_vol_cat_filter}
      {_vol_sch_filter}
""").collect()

print(f"Total de volumes encontrados: {len(volume_list)}")

# --- Cálculo de tamanho por volume (recursivo) ---

volume_sizes = []
for vol in volume_list:
    vol_path = f"/Volumes/{vol.volume_catalog}/{vol.volume_schema}/{vol.volume_name}"
    total_size, file_count = _get_volume_size_recursive(vol_path)
    volume_sizes.append({
        "catalog": vol.volume_catalog,
        "schema": vol.volume_schema,
        "volume": vol.volume_name,
        "volume_type": vol.volume_type,
        "storage_location": vol.storage_location,
        "total_size_bytes": total_size,
        "file_count": file_count,
        "created_at": str(vol.created_at)[:19] if vol.created_at else None,
        "last_altered_at": str(vol.last_altered_at)[:19] if vol.last_altered_at else None,
    })

# --- Resultado ---

if volume_sizes:
    df_volumes = spark.createDataFrame(volume_sizes) \
        .withColumn("size_gb", spark_round(col("total_size_bytes") / (1024**3), 2)) \
        .orderBy(col("total_size_bytes").desc()) \
        .limit(TOP_N)
    df_volumes.createOrReplaceTempView("volume_sizes")
    display(df_volumes.select("catalog", "schema", "volume", "volume_type", "size_gb", "file_count", "last_altered_at"))
else:
    print("⚠️ Nenhum volume encontrado com os filtros configurados.")

## 4. Objetos Obsoletos (Stale)
Identifica tabelas e volumes que não foram acessados ou modificados nos últimos N dias (configurável). Objetos inativos que ainda consomem storage são candidatos a arquivamento ou remoção.

In [0]:
# Detecta tabelas sem acesso/modificação recente usando system.access.audit
# e informações de last_altered do information_schema

from pyspark.sql.functions import col, datediff, current_date, lit, when
from datetime import datetime, timedelta

cutoff_date = (datetime.now() - timedelta(days=STALE_DAYS_THRESHOLD)).strftime("%Y-%m-%d")

# Abordagem 1: Usar last_altered do information_schema
try:
    stale_query = f"""
    SELECT 
        table_catalog,
        table_schema,
        table_name,
        table_type,
        data_source_format,
        created,
        last_altered,
        DATEDIFF(CURRENT_DATE(), last_altered) AS days_since_last_change
    FROM system.information_schema.tables
    WHERE table_type IN ('MANAGED', 'EXTERNAL')
      AND last_altered < '{cutoff_date}'
      {catalog_clause}
      {schema_clause}
    ORDER BY last_altered ASC
    LIMIT {TOP_N}
    """
    
    df_stale = spark.sql(stale_query)
    stale_count = df_stale.count()
    
    print(f"⚠️ {stale_count} tabelas sem alteração nos últimos {STALE_DAYS_THRESHOLD} dias:")
    print(f"   Data de corte: {cutoff_date}")
    print(f"   Essas tabelas podem ser candidatas a arquivamento ou deleção.\n")
    
    df_stale.createOrReplaceTempView("stale_tables")
    display(df_stale)

except Exception as e:
    print(f"⚠️ Erro na detecção de objetos stale: {e}")

In [0]:
# Complementa a análise anterior com dados de acesso REAL via audit logs.
# Usa generateTemporaryTableCredential como sinal — esse evento ocorre quando
# o runtime solicita credenciais para ler/escrever arquivos da tabela no storage.
# Isso exclui acessos puramente administrativos (DESCRIBE, SHOW, ALTER, etc.)
#
# NOTA: O campo correto é request_params['table_full_name'] (NÃO full_name_arg,
# que é NULL para este action_name).

from datetime import datetime, timedelta

catalog_clause = f"AND table_catalog = '{CATALOG_FILTER}'" if CATALOG_FILTER else ""
schema_clause = f"AND table_schema = '{SCHEMA_FILTER}'" if SCHEMA_FILTER else ""

cutoff_date = (datetime.now() - timedelta(days=STALE_DAYS_THRESHOLD)).strftime("%Y-%m-%d")

try:
    access_query = f"""
    WITH table_access AS (
        -- Agrega eventos de acesso real ao storage por tabela.
        -- Exige acesso em mais de 1 dia distinto para considerar como "uso ativo".
        -- Isso exclui Predictive Optimization (VACUUM/OPTIMIZE automático)
        -- que tipicamente toca todas as tabelas managed em um único dia.
        SELECT 
            request_params['table_full_name'] AS table_full_name,
            MAX(event_date) AS last_access_date,
            COUNT(*) AS access_count,
            COUNT(DISTINCT event_date) AS distinct_access_days
        FROM system.access.audit
        WHERE action_name = 'generateTemporaryTableCredential'
          AND event_date >= '{cutoff_date}'
          AND request_params['table_full_name'] IS NOT NULL
        GROUP BY request_params['table_full_name']
        HAVING COUNT(DISTINCT event_date) > 1  -- Exclui acesso em dia único (provavel PO)
    ),
    all_tables AS (
        SELECT 
            CONCAT(table_catalog, '.', table_schema, '.', table_name) AS table_full_name,
            table_catalog,
            table_schema,
            table_name,
            created,
            last_altered
        FROM system.information_schema.tables
        WHERE table_type IN ('MANAGED', 'EXTERNAL')
          {catalog_clause}
          {schema_clause}
    )
    SELECT 
        t.table_catalog,
        t.table_schema,
        t.table_name,
        t.created,
        t.last_altered,
        a.last_access_date,
        COALESCE(a.access_count, 0) AS total_access_events,
        COALESCE(a.distinct_access_days, 0) AS distinct_access_days,
        CASE 
            WHEN a.table_full_name IS NULL THEN '🛑 SEM USO ATIVO no período'
            ELSE '✅ Uso ativo (multi-dia)'
        END AS access_status
    FROM all_tables t
    LEFT JOIN table_access a ON t.table_full_name = a.table_full_name
    WHERE a.table_full_name IS NULL
    ORDER BY t.last_altered ASC
    LIMIT {TOP_N}
    """
    
    df_no_access = spark.sql(access_query)
    no_access_count = df_no_access.count()
    
    print(f"🛑 {no_access_count} tabelas sem uso ativo nos últimos {STALE_DAYS_THRESHOLD} dias (via audit logs):")
    print(f"   Sinal: generateTemporaryTableCredential com acesso em >1 dia distinto")
    print(f"   (exclui Predictive Optimization que toca tabelas em dia único)")
    print(f"   Estas são as candidatas para arquivamento/remoção.\n")
    
    df_no_access.createOrReplaceTempView("tables_no_access")
    display(df_no_access)

except Exception as e:
    print(f"⚠️ Não foi possível acessar audit logs: {e}")
    print("Verifique se você tem permissão para system.access.audit")
    print("A análise de stale tables acima (baseada em last_altered) continua válida.")

## 5. Objetos Temporários e Intermediários
Detecta tabelas cujo nome sugere caráter temporário (tmp, staging, backup, test, etc.) que podem ter sido esquecidas após o processamento.

In [0]:
# Identifica tabelas com nomes que sugerem caráter temporário

from pyspark.sql.functions import col, lower, array, lit, expr
import re

# Construir cláusula de filtro para padrões temporários
# Usa CONTAINS() para match literal (evita que _ seja interpretado como wildcard do LIKE)
pattern_conditions = " OR ".join(
    [f"CONTAINS(LOWER(table_name), '{p}')" for p in TEMP_PATTERNS]
)

try:
    temp_query = f"""
    SELECT 
        table_catalog,
        table_schema,
        table_name,
        table_type,
        data_source_format,
        created,
        last_altered,
        DATEDIFF(CURRENT_DATE(), last_altered) AS days_since_change,
        CASE
            {' '.join([f"WHEN CONTAINS(LOWER(table_name), '{p}') THEN '{p}'" for p in TEMP_PATTERNS])}
            ELSE 'outro'
        END AS pattern_matched
    FROM system.information_schema.tables
    WHERE table_type IN ('MANAGED', 'EXTERNAL')
      AND ({pattern_conditions})
      {catalog_clause}
      {schema_clause}
    ORDER BY last_altered ASC
    LIMIT {TOP_N}
    """
    
    df_temp = spark.sql(temp_query)
    temp_count = df_temp.count()
    
    print(f"⚠️ {temp_count} tabelas com nomes que sugerem caráter temporário/intermediário:")
    print(f"   Padrões procurados: {TEMP_PATTERNS}")
    print(f"   Tabelas mais antigas sem alteração são as mais prováveis de serem desnecessárias.\n")
    
    df_temp.createOrReplaceTempView("temp_tables")
    display(df_temp)

except Exception as e:
    print(f"⚠️ Erro na detecção de tabelas temporárias: {e}")

## 6. Possíveis Duplicidades entre Camadas/Ambientes
Detecta tabelas com nomes similares em schemas diferentes, que podem representar duplicações entre camadas (bronze/silver/gold) ou ambientes (dev/staging/prod).

In [0]:
# Identifica tabelas com mesmo nome em diferentes schemas/catálogos
# Possíveis duplicatas entre camadas bronze/silver/gold ou entre ambientes

from pyspark.sql.functions import col, count, collect_list, concat_ws, size

try:
    dup_query = f"""
    SELECT 
        table_name,
        COUNT(*) AS occurrences,
        COLLECT_SET(CONCAT(table_catalog, '.', table_schema)) AS locations,
        COLLECT_SET(table_type) AS table_types,
        COLLECT_SET(data_source_format) AS formats
    FROM system.information_schema.tables
    WHERE table_type IN ('MANAGED', 'EXTERNAL')
      {catalog_clause}
    GROUP BY table_name
    HAVING COUNT(*) > 1
    ORDER BY COUNT(*) DESC
    LIMIT {TOP_N}
    """
    
    df_duplicates = spark.sql(dup_query)
    dup_count = df_duplicates.count()
    
    print(f"⚠️ {dup_count} nomes de tabela que aparecem em múltiplos schemas:")
    print(f"   Verifique se são duplicações reais ou se há razão legítima para existências múltiplas.")
    print(f"   Padrões comuns de duplicidade:")
    print(f"   - Mesma tabela em bronze + silver + gold (sem drop da camada anterior)")
    print(f"   - Cópias entre dev e prod esquecidas")
    print(f"   - Backups manuais com mesmo nome em outro schema\n")
    
    df_duplicates.createOrReplaceTempView("duplicate_tables")
    display(df_duplicates)

except Exception as e:
    print(f"⚠️ Erro na detecção de duplicatas: {e}")

In [0]:
# Identifica schemas que contêm camadas sobrepostas
# Ex: schemas "bronze", "silver", "gold" com tabelas de mesmo nome

layer_patterns = ['bronze', 'silver', 'gold', 'raw', 'curated', 'refined', 
                  'landing', 'staging', 'dev', 'test', 'prod', 'sandbox']

try:
    layer_conditions = " OR ".join(
        [f"LOWER(table_schema) LIKE '%{p}%'" for p in layer_patterns]
    )
    
    overlap_query = f"""
    WITH layered_tables AS (
        SELECT 
            table_catalog,
            table_schema,
            table_name,
            CASE
                {' '.join([f"WHEN LOWER(table_schema) LIKE '%{p}%' THEN '{p}'" for p in layer_patterns])}
                ELSE 'other'
            END AS detected_layer
        FROM system.information_schema.tables
        WHERE table_type IN ('MANAGED', 'EXTERNAL')
          AND ({layer_conditions})
          {catalog_clause}
    )
    SELECT 
        table_name,
        COUNT(DISTINCT detected_layer) AS num_layers,
        COLLECT_SET(detected_layer) AS layers_present,
        COLLECT_SET(CONCAT(table_catalog, '.', table_schema)) AS full_locations
    FROM layered_tables
    WHERE detected_layer != 'other'
    GROUP BY table_name
    HAVING COUNT(DISTINCT detected_layer) > 1
    ORDER BY num_layers DESC, table_name
    LIMIT {TOP_N}
    """
    
    df_layer_overlap = spark.sql(overlap_query)
    overlap_count = df_layer_overlap.count()
    
    if overlap_count > 0:
        print(f"🔄 {overlap_count} tabelas presentes em múltiplas camadas do lakehouse:")
        print(f"   Verifique se as camadas intermediárias ainda são necessárias.\n")
        display(df_layer_overlap)
    else:
        print("✅ Nenhuma sobreposição explícita detectada entre camadas nomeadas.")

except Exception as e:
    print(f"⚠️ Erro na análise de camadas: {e}")

## 7. Streaming Checkpoints, Logs e Artefatos de Processamento
Identifica artefatos de Structured Streaming (checkpoints, offsets) e tabelas de log que crescem indefinidamente sem política de retenção.

In [0]:
# Identifica tabelas e artefatos relacionados a streaming, checkpoints e logs

log_patterns = ['checkpoint', 'ckpt', '_log', 'logs', 'audit', 'offset', 
                'streaming', '_events', '_metrics', '_monitor']

try:
    log_conditions = " OR ".join(
        [f"LOWER(table_name) LIKE '%{p}%'" for p in log_patterns]
    )
    
    log_query = f"""
    SELECT 
        table_catalog,
        table_schema,
        table_name,
        table_type,
        data_source_format,
        created,
        last_altered,
        DATEDIFF(CURRENT_DATE(), created) AS age_days,
        CASE
            {' '.join([f"WHEN LOWER(table_name) LIKE '%{p}%' THEN '{p}'" for p in log_patterns])}
            ELSE 'outro'
        END AS artifact_type
    FROM system.information_schema.tables
    WHERE table_type IN ('MANAGED', 'EXTERNAL')
      AND ({log_conditions})
      {catalog_clause}
      {schema_clause}
    ORDER BY last_altered ASC
    LIMIT {TOP_N}
    """
    
    df_logs = spark.sql(log_query)
    log_count = df_logs.count()
    
    print(f"📝 {log_count} tabelas/artefatos de logging/checkpoint encontrados:")
    print(f"   Padrões procurados: {log_patterns}")
    print(f"   Tabelas de log sem política de retenção podem crescer indefinidamente.")
    print(f"   Checkpoints antigos de streaming já concluído podem ser removidos.\n")
    
    df_logs.createOrReplaceTempView("log_artifacts")
    display(df_logs)

except Exception as e:
    print(f"⚠️ Erro na detecção de artefatos de log: {e}")

In [0]:
# Verifica checkpoints de streaming armazenados em volumes ou paths do DBFS
# Checkpoints antigos de jobs que não rodam mais consomem storage desnecessariamente

checkpoint_paths_found = []

try:
    # Verificar se existem volumes com padrões de checkpoint
    _ckpt_cat_filter = f"AND volume_catalog = '{CATALOG_FILTER}'" if CATALOG_FILTER else ""
    checkpoint_volumes = spark.sql(f"""
        SELECT 
            volume_catalog,
            volume_schema,
            volume_name,
            storage_location
        FROM system.information_schema.volumes
        WHERE (LOWER(volume_name) LIKE '%checkpoint%'
           OR LOWER(volume_name) LIKE '%ckpt%'
           OR LOWER(volume_name) LIKE '%streaming%')
        {_ckpt_cat_filter}
    """).collect()
    
    if checkpoint_volumes:
        print(f"📦 {len(checkpoint_volumes)} volumes com padrão de checkpoint encontrados:")
        for vol in checkpoint_volumes:
            vol_path = f"/Volumes/{vol.volume_catalog}/{vol.volume_schema}/{vol.volume_name}"
            try:
                items = dbutils.fs.ls(vol_path)
                total_size = sum(f.size for f in items if f.size)
                print(f"   • {vol.volume_catalog}.{vol.volume_schema}.{vol.volume_name}")
                print(f"     Localização: {vol.storage_location}")
                print(f"     Arquivos: {len(items)} | Tamanho: {total_size / (1024**3):.2f} GB")
            except:
                print(f"   • {vol.volume_catalog}.{vol.volume_schema}.{vol.volume_name} (sem acesso)")
    else:
        print("✅ Nenhum volume com padrão de checkpoint encontrado.")
        
except Exception as e:
    print(f"⚠️ Erro ao verificar checkpoints em volumes: {e}")

## 8. Diagnóstico de Políticas de Retenção
Verifica quais tabelas possuem (ou não) configurações explícitas de retenção:
- `delta.deletedFileRetentionDuration` (padrão: 7 dias)
- `delta.logRetentionDuration` (padrão: 30 dias)

Tabelas grandes SEM política explícita podem estar acumulando histórico desnecessário.

In [0]:
# Verifica políticas de retenção configuradas nas tabelas Delta
# Tabelas grandes sem política explícita são as mais problématicas

retention_results = []

for row in table_list:
    fqn = f"`{row.table_catalog}`.`{row.table_schema}`.`{row.table_name}`"
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {fqn}").collect()
        props_dict = {p.key: p.value for p in props}
        
        deleted_retention = props_dict.get('delta.deletedFileRetentionDuration', 'DEFAULT (7 days)')
        log_retention = props_dict.get('delta.logRetentionDuration', 'DEFAULT (30 days)')
        enable_change_feed = props_dict.get('delta.enableChangeDataFeed', 'false')
        
        # Verificar se há propriedades de retenção customizadas
        has_custom_retention = (
            'delta.deletedFileRetentionDuration' in props_dict or
            'delta.logRetentionDuration' in props_dict
        )
        
        retention_results.append({
            "catalog": row.table_catalog,
            "schema": row.table_schema,
            "table": row.table_name,
            "deleted_file_retention": deleted_retention,
            "log_retention": log_retention,
            "change_data_feed": enable_change_feed,
            "has_custom_retention": has_custom_retention,
            "status": "✅ Configurada" if has_custom_retention else "⚠️ Sem política explícita"
        })
    except Exception:
        pass

if retention_results:
    df_retention = spark.createDataFrame(retention_results)
    
    # Separar tabelas sem política
    df_no_policy = df_retention.filter(col("has_custom_retention") == False)
    df_with_policy = df_retention.filter(col("has_custom_retention") == True)
    
    no_policy_count = df_no_policy.count()
    with_policy_count = df_with_policy.count()
    
    print(f"\n📊 Resumo de Políticas de Retenção:")
    print(f"   ✅ Tabelas COM política explícita: {with_policy_count}")
    print(f"   ⚠️ Tabelas SEM política explícita: {no_policy_count}")
    print(f"")
    print(f"   Tabelas sem política usam os defaults do Delta:")
    print(f"   - deletedFileRetentionDuration = 7 dias")
    print(f"   - logRetentionDuration = 30 dias")
    print(f"   Para tabelas grandes com muitas escritas, considere reduzir esses valores.\n")
    
    df_retention.createOrReplaceTempView("retention_policies")
    display(df_no_policy.limit(TOP_N))
else:
    print("⚠️ Não foi possível verificar políticas de retenção.")

## 9. Resumo Consolidado e Recomendações
Consolida todos os achados e gera recomendações acionáveis.

In [0]:
# =============================================================================
# RESUMO CONSOLIDADO - Consolida todos os achados das seções anteriores
# =============================================================================

from pyspark.sql.functions import col, sum as spark_sum, count as spark_count, lit

print("""
╔════════════════════════════════════════════════════════════╗
║        STORAGE DIAGNOSTICS - RESUMO FINAL              ║
╚════════════════════════════════════════════════════════════╝
""")

# Coletar métricas das seções anteriores
findings = []

# 1. Tabelas grandes
try:
    big_tables = spark.sql(f"""
        SELECT COUNT(*) as cnt, COALESCE(SUM(size_gb), 0) as total_gb 
        FROM table_sizes_detail 
        WHERE size_gb > {TABLE_SIZE_THRESHOLD_GB}
    """).collect()[0]
    findings.append({
        "categoria": "1. Tabelas Grandes",
        "achados": f"{big_tables.cnt} tabelas acima de {TABLE_SIZE_THRESHOLD_GB} GB",
        "impacto_gb": float(big_tables.total_gb),
        "prioridade": "🔴 ALTA" if big_tables.cnt > 0 else "🟢 OK"
    })
except:
    findings.append({"categoria": "1. Tabelas Grandes", "achados": "N/A (execute seção 1)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 2. Bloat de versões
try:
    bloat_count = spark.sql("SELECT COUNT(*) as cnt FROM version_bloat").collect()[0].cnt
    findings.append({
        "categoria": "2. Bloat de Versões Delta",
        "achados": f"{bloat_count} tabelas com >{VERSION_COUNT_THRESHOLD} versões",
        "impacto_gb": -1.0,  # Difícil estimar sem vacuum dry run
        "prioridade": "🔴 ALTA" if bloat_count > 5 else "🟡 MÉDIA" if bloat_count > 0 else "🟢 OK"
    })
except:
    findings.append({"categoria": "2. Bloat de Versões Delta", "achados": "N/A (execute seção 2)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 3. Volumes
try:
    vol_stats = spark.sql("SELECT COUNT(*) as cnt, COALESCE(SUM(size_gb), 0) as total_gb FROM volume_sizes WHERE size_gb > 1").collect()[0]
    findings.append({
        "categoria": "3. Volumes Pesados",
        "achados": f"{vol_stats.cnt} volumes com >1 GB",
        "impacto_gb": float(vol_stats.total_gb),
        "prioridade": "🔴 ALTA" if vol_stats.total_gb > 100 else "🟡 MÉDIA" if vol_stats.cnt > 0 else "🟢 OK"
    })
except:
    findings.append({"categoria": "3. Volumes Pesados", "achados": "N/A (execute seção 3)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 4. Objetos stale
try:
    stale_count = spark.sql("SELECT COUNT(*) as cnt FROM stale_tables").collect()[0].cnt
    findings.append({
        "categoria": "4. Objetos Obsoletos (Stale)",
        "achados": f"{stale_count} tabelas sem alteração em >{STALE_DAYS_THRESHOLD} dias",
        "impacto_gb": -1.0,
        "prioridade": "🔴 ALTA" if stale_count > 20 else "🟡 MÉDIA" if stale_count > 0 else "🟢 OK"
    })
except:
    findings.append({"categoria": "4. Objetos Obsoletos (Stale)", "achados": "N/A (execute seção 4)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 5. Temporários
try:
    temp_count = spark.sql("SELECT COUNT(*) as cnt FROM temp_tables").collect()[0].cnt
    findings.append({
        "categoria": "5. Objetos Temporários",
        "achados": f"{temp_count} tabelas com nomes temporários",
        "impacto_gb": -1.0,
        "prioridade": "🟡 MÉDIA" if temp_count > 10 else "🟢 OK" if temp_count == 0 else "🟡 BAIXA"
    })
except:
    findings.append({"categoria": "5. Objetos Temporários", "achados": "N/A (execute seção 5)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 6. Duplicatas
try:
    dup_count = spark.sql("SELECT COUNT(*) as cnt FROM duplicate_tables").collect()[0].cnt
    findings.append({
        "categoria": "6. Possíveis Duplicatas",
        "achados": f"{dup_count} nomes duplicados entre schemas",
        "impacto_gb": -1.0,
        "prioridade": "🟡 MÉDIA" if dup_count > 10 else "🟢 OK" if dup_count == 0 else "🟡 BAIXA"
    })
except:
    findings.append({"categoria": "6. Possíveis Duplicatas", "achados": "N/A (execute seção 6)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 7. Logs e Checkpoints
try:
    log_count = spark.sql("SELECT COUNT(*) as cnt FROM log_artifacts").collect()[0].cnt
    findings.append({
        "categoria": "7. Logs e Checkpoints",
        "achados": f"{log_count} artefatos de log/checkpoint encontrados",
        "impacto_gb": -1.0,
        "prioridade": "🟡 MÉDIA" if log_count > 10 else "🟢 OK" if log_count == 0 else "🟡 BAIXA"
    })
except:
    findings.append({"categoria": "7. Logs e Checkpoints", "achados": "N/A (execute seção 7)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# 8. Retenção
try:
    no_policy = spark.sql("SELECT COUNT(*) as cnt FROM retention_policies WHERE has_custom_retention = false").collect()[0].cnt
    findings.append({
        "categoria": "8. Sem Política de Retenção",
        "achados": f"{no_policy} tabelas sem política explícita",
        "impacto_gb": -1.0,
        "prioridade": "🟡 MÉDIA" if no_policy > 20 else "🟢 OK" if no_policy == 0 else "🟡 BAIXA"
    })
except:
    findings.append({"categoria": "8. Sem Política de Retenção", "achados": "N/A (execute seção 8)", "impacto_gb": 0.0, "prioridade": "⭕ N/A"})

# Exibir resumo
df_summary = spark.createDataFrame(findings)
display(df_summary.select("prioridade", "categoria", "achados"))

In [0]:
# =============================================================================
# RECOMENDAÇÕES DE AÇÃO
# =============================================================================

print("""
┌────────────────────────────────────────────────────────────┐
│  🛠️  RECOMENDAÇÕES DE AÇÃO                            │
└────────────────────────────────────────────────────────────┘

━━━ AÇÕES IMEDIATAS (Quick Wins) ━━━

1️⃣  VACUUM em tabelas com bloat de versões:
   • Execute VACUUM <tabela> RETAIN 168 HOURS para cada tabela identificada
   • Para automação: habilite Predictive Optimization no Unity Catalog
     (executa VACUUM e OPTIMIZE automaticamente)

2️⃣  Remover tabelas temporárias obsoletas:
   • Revise a lista da seção 5 com os donos dos schemas
   • DROP TABLE após confirmação (não pode ser desfeito!)

3️⃣  Limpar volumes com arquivos desnecessários:
   • Verifique uploads antigos e exports esquecidos


━━━ AÇÕES DE MÉDIO PRAZO ━━━

4️⃣  Implementar políticas de retenção:
   • ALTER TABLE <t> SET TBLPROPERTIES (
       'delta.deletedFileRetentionDuration' = 'interval 7 days',
       'delta.logRetentionDuration' = 'interval 30 days'
     )
   • Para tabelas com alta frequência de escrita, considere retenção menor

5️⃣  Arquivar ou dropar tabelas stale:
   • Tabelas sem acesso em >90 dias devem ser revisadas com stakeholders
   • Considere mover para cold storage antes de deletar

6️⃣  Consolidar duplicatas entre camadas:
   • Verifique se camadas intermediárias (bronze) ainda são necessárias
   • Views podem substituir cópias físicas em muitos casos


━━━ AÇÕES ESTRATÉGICAS (Prevenção) ━━━

7️⃣  Habilitar Predictive Optimization:
   • ALTER CATALOG <c> ENABLE PREDICTIVE OPTIMIZATION
   • Automatiza VACUUM e OPTIMIZE para todas as tabelas managed

8️⃣  Implementar governance de lifecycle:
   • Tags obrigatórias: owner, retention_days, environment
   • Processo de revisão periódica de storage (ex: mensal)
   • Alertas automáticos quando tabelas excedem threshold de tamanho

9️⃣  Automatizar limpeza:
   • Job agendado executando este notebook semanalmente
   • Integrar com notificações (email/Slack) para achados críticos
   • Script de VACUUM automatizado para tabelas sem Predictive Optimization

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⚠️  IMPORTANTE: Sempre valide com os donos dos dados antes de deletar!
   Este notebook identifica CANDIDATOS, a decisão final é humana.
""")